In [9]:
# Core imports
import numpy as np
import pandas as pd
import re
import ast
import json
from pathlib import Path

# Text processing
import spacy
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

# Scikit‑learn utilities
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

# Persistence & progress
import joblib
from tqdm.auto import tqdm

# Suppress warnings that are not useful in a notebook
import warnings
warnings.filterwarnings('ignore')


In [10]:
# ------------------------------------------------------------------
# 1️⃣ Load data
# ------------------------------------------------------------------
# Adjust the paths if you placed the CSVs somewhere else.
movies = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')

# Handy preview – you can delete this cell later.
movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [11]:
# ------------------------------------------------------------------
# 2️⃣ Parse JSON columns (genres, keywords, cast, crew)
# ------------------------------------------------------------------
def literal_eval_series(series):
    """Convert a pandas Series of stringified lists/dicts to python objects safely."""
    return series.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

# Genres – list of dicts → list of genre names
movies['genres'] = literal_eval_series(movies['genres']).apply(lambda lst: [d['name'] for d in lst])

# Keywords – list of dicts → list of keyword names
movies['keywords'] = literal_eval_series(movies['keywords']).apply(lambda lst: [d['name'] for d in lst])

# Cast – keep first 5 actors (names only)
credits['cast'] = literal_eval_series(credits['cast']).apply(
    lambda lst: [c['name'] for c in lst][:5]
)

# Crew – keep only director names
credits['crew'] = literal_eval_series(credits['crew']).apply(
    lambda lst: [c['name'] for c in lst if c.get('job') == 'Director']
)

# Rename the identifier column in movies so it matches credits
movies = movies.rename(columns={'id': 'movie_id'})

# Merge the needed columns from credits into movies
movies = movies.merge(credits[['movie_id', 'cast', 'crew']], on='movie_id', how='left')

# Year (already present) – keep as int
movies['year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year.fillna(0).astype(int)

# Keep only the columns we will use downstream
new_df = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'year', 'vote_average']].copy()


In [12]:
# ------------------------------------------------------------------
# 3️⃣ Build a single "tags" column – everything the model sees.
# ------------------------------------------------------------------
# Replace missing values with empty strings/lists to avoid errors later
for col in ['overview', 'genres', 'keywords', 'cast', 'crew']:
    new_df[col] = new_df[col].apply(lambda x: x if isinstance(x, (list, str)) else "")

# Concatenate everything into one big string (list elements are joined by spaces)
def list_to_str(x):
    if isinstance(x, list):
        return " ".join(map(str, x))
    return str(x)

new_df['tags'] = (
    new_df['overview'].apply(list_to_str) + " " +
    new_df['genres'].apply(list_to_str) + " " +
    new_df['keywords'].apply(list_to_str) + " " +
    new_df['cast'].apply(list_to_str) + " " +
    new_df['crew'].apply(list_to_str)
)

# Lower‑case & strip punctuation (kept simple for speed)
new_df['tags'] = new_df['tags'].str.lower().str.replace(r'[^a-z0-9 ]', ' ', regex=True)

# Lemmatise – spaCy is faster and more accurate than Porter stemming
def lemmatise(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc if token.is_alpha])

tqdm.pandas(desc='Lemmatising tags')
new_df['tags'] = new_df['tags'].progress_apply(lemmatise)

# Show a preview – optional
new_df[['title', 'tags']].head()

Lemmatising tags:   0%|          | 5/4803 [00:00<03:45, 21.32it/s]

Lemmatising tags: 100%|██████████| 4803/4803 [01:28<00:00, 54.56it/s]


,title,tags
0,Avatar,in the century a paraplegic marine be dispatch...
1,Pirates of the Caribbean: At World's End,captain barbossa long believe to be dead have ...
2,Spectre,a cryptic message from bond s past send he on ...
3,The Dark Knight Rises,follow the death of district attorney harvey d...
4,John Carter,john carter be a war weary former military cap...


In [13]:
# ------------------------------------------------------------------
# 4️⃣ Vectorise – TF‑IDF + Truncated SVD (LSA)
# ------------------------------------------------------------------
# TF‑IDF (unigrams & bigrams, up to 20 000 features, sub‑linear TF‑IDF)
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words='english',
    sublinear_tf=True
)

# Sparse TF‑IDF matrix (do NOT convert to dense!)
tfidf_matrix = vectorizer.fit_transform(new_df['tags'])

# Dimensionality reduction – keep 300 latent topics (good trade‑off)
svd = TruncatedSVD(n_components=300, random_state=42)
reduced_matrix = svd.fit_transform(tfidf_matrix)

# Normalise rows to unit length – improves cosine NN queries
norms = np.linalg.norm(reduced_matrix, axis=1, keepdims=True)
reduced_matrix = reduced_matrix / np.where(norms == 0, 1, norms)

# Build a fast cosine‑NN index (brute‑force works fine for 5 k items)
nn = NearestNeighbors(metric='cosine', algorithm='brute')
nn.fit(reduced_matrix)

# Store everything that is needed to serve a model later
model_bundle = {
    'vectorizer': vectorizer,
    'svd': svd,
    'nn': nn,
    'df': new_df.reset_index(drop=True)   # will be used for lookup
}

# Quick sanity‑check – distance of a movie to itself should be ~0
sample_idx = 0
dist, _ = nn.kneighbors(reduced_matrix[[sample_idx]], n_neighbors=1)
print('Self‑distance (should be ~0):', dist[0][0])

Self‑distance (should be ~0): 0.0


In [14]:
# ------------------------------------------------------------------
# 5️⃣ Recommendation function – flexible & hybrid
# ------------------------------------------------------------------
def recommend(movie_title: str, top_n: int = 10, min_rating: float = None, min_year: int = None):
    """Return the *top_n* most similar movies to *movie_title*.
    Optional filters:
      * min_rating – only return movies with vote_average >= this value.
      * min_year   – only return movies released in or after this year.
    The function also returns a *score* that is a weighted hybrid of
    content similarity (0.7) and popularity (vote_average – 0.3).
    """

    # ------------------------------------------------------------------
    # 1️⃣ Find the index of the query movie in the original DataFrame
    # ------------------------------------------------------------------
    df = model_bundle['df']
    if movie_title not in df['title'].values:
        raise ValueError(f"Movie '{movie_title}' not found in the dataset.")
    query_idx = df.index[df['title'] == movie_title][0]

    # ------------------------------------------------------------------
    # 2️⃣ Retrieve the reduced‑dimensional vector for the query movie
    # ------------------------------------------------------------------
    query_vec = model_bundle['nn']._fit_X[query_idx].reshape(1, -1)

    # ------------------------------------------------------------------
    # 3️⃣ Perform a cosine‑NN search – ask for a few extra neighbours
    # ------------------------------------------------------------------
    distances, indices = model_bundle['nn'].kneighbors(query_vec, n_neighbors=top_n + 15)
    distances = distances.ravel()
    indices = indices.ravel()

    # ------------------------------------------------------------------
    # 4️⃣ Build a result DataFrame, filter, and compute hybrid scores
    # ------------------------------------------------------------------
    results = df.iloc[indices].copy()
    results['cosine_dist'] = distances
    # Convert cosine distance to similarity (higher is better)
    results['content_sim'] = 1 - results['cosine_dist']

    # Optional filters
    if min_rating is not None:
        results = results[results['vote_average'] >= min_rating]
    if min_year is not None:
        results = results[results['year'] >= min_year]

    # Hybrid re‑ranking – 0.7 content + 0.3 normalised rating
    max_rating = df['vote_average'].max()
    results['rating_norm'] = results['vote_average'] / max_rating
    results['hybrid_score'] = 0.7 * results['content_sim'] + 0.3 * results['rating_norm']

    # Drop the query movie itself and keep the top_n rows
    results = results[results['title'] != movie_title]
    results = results.sort_values('hybrid_score', ascending=False).head(top_n)

    # Return only the columns we need for display
    return results[['title', 'year', 'vote_average', 'hybrid_score']]

# ------------------------------------------------------------------
# 6️⃣ Demo – ask for 10 recommendations for "Spider‑Man 2"
# ------------------------------------------------------------------
recommend('Spider-Man 2', top_n=10, min_rating=5.0, min_year=2000)

,title,year,vote_average,hybrid_score
5,Spider-Man 3,2007,5.9,0.774463
159,Spider-Man,2002,6.8,0.722352
20,The Amazing Spider-Man,2012,6.5,0.562685
38,The Amazing Spider-Man 2,2014,6.5,0.552042
788,Deadpool,2016,7.4,0.497223
119,Batman Begins,2005,7.5,0.428308
7,Avengers: Age of Ultron,2015,7.3,0.416605
182,Ant-Man,2015,7.0,0.415375
1740,Kick-Ass 2,2013,6.3,0.407369
215,Fantastic 4: Rise of the Silver Surfer,2007,5.4,0.392976


In [15]:
# ------------------------------------------------------------------
# 7️⃣ Persist the whole pipeline (so you can load it later in a web‑app)
# ------------------------------------------------------------------
Path('artifacts').mkdir(parents=True, exist_ok=True)
joblib.dump(model_bundle, 'artifacts/recommender.pkl', compress=('gzip', 3))
print('Model saved to artifacts/recommender.pkl')

Model saved to artifacts/recommender.pkl


In [16]:
# ------------------------------------------------------------------
# 8️⃣ (Optional) Quick evaluation – Recall@k on a random hold‑out set
# ------------------------------------------------------------------
from sklearn.model_selection import train_test_split

# Split the dataset (80 % train, 20 % test) – we’ll use the *train* part to build the index
df_train, df_test = train_test_split(new_df, test_size=0.2, random_state=42)

# Re‑build the pipeline only on the training part (to mimic a real production scenario)
tfidf_train = vectorizer.fit_transform(df_train['tags'])
reduced_train = svd.fit_transform(tfidf_train)
norms = np.linalg.norm(reduced_train, axis=1, keepdims=True)
reduced_train = reduced_train / np.where(norms == 0, 1, norms)
nn_train = NearestNeighbors(metric='cosine', algorithm='brute').fit(reduced_train)

# Helper – get top‑k neighbours for a given row index (in *df_test*)
def get_top_k(idx_test, k=10):
    vec = svd.transform(vectorizer.transform([df_test.iloc[idx_test]['tags']]))
    vec = vec / np.linalg.norm(vec) if np.linalg.norm(vec) != 0 else vec
    d, i = nn_train.kneighbors(vec, n_neighbors=k + 1)   # +1 because the query itself may appear
    i = i[0]
    d = d[0]
    if i[0] == idx_test:
        i = i[1:]
        d = d[1:]
    else:
        i = i[:k]
        d = d[:k]
    return i, d

# Compute Recall@k (k = 5,10,20)
def recall_at_k(k):
    hits = 0
    for idx in range(len(df_test)):
        neigh_idx, _ = get_top_k(idx, k)
        # In a pure content‑based scenario the target is the *same* title – we check if it appears in the neighbour set
        if df_test.iloc[idx]['title'] in df_train.iloc[neigh_idx]['title'].values:
            hits += 1
    return hits / len(df_test)

for k in [5, 10, 20]:
    print(f"Recall@{k}: {recall_at_k(k):.3f}")

Recall@5: 0.000
Recall@10: 0.000
Recall@20: 0.001
